# ModelForge Lite — Phase 5: Fine-Tuned Model + RAG

This notebook loads the base model, attaches the LoRA adapter from Phase 4, and adds the same retrieval pipeline from Phase 3 on top. This is Variant 4 of 4 — the combination the whole project is building toward.

Run in Colab with the T4 GPU enabled.

In [ ]:
!pip install -q transformers accelerate datasets huggingface_hub pandas peft sentence-transformers faiss-cpu

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 1. Pull the knowledge base and eval split

In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download

HF_USERNAME = "YOUR_HF_USERNAME"
dataset_repo = f"{HF_USERNAME}/modelforge-lite-support-data"
adapter_repo = f"{HF_USERNAME}/modelforge-lite-lora-adapter"

kb_path = hf_hub_download(repo_id=dataset_repo, filename="knowledge_base.csv", repo_type="dataset")
eval_path = hf_hub_download(repo_id=dataset_repo, filename="eval.csv", repo_type="dataset")

kb_df = pd.read_csv(kb_path)
eval_df = pd.read_csv(eval_path)
print(f"Knowledge base: {len(kb_df)} rows | Eval set: {len(eval_df)} rows")

## 2. Build the FAISS index (same as Phase 3)

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

kb_texts = (kb_df["instruction"] + " " + kb_df["response"]).tolist()
kb_embeddings = embedder.encode(kb_texts, show_progress_bar=True, convert_to_numpy=True)

index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)
print(f"FAISS index built with {index.ntotal} vectors")

def retrieve(question, k=3):
    q_embedding = embedder.encode([question], convert_to_numpy=True)
    distances, indices = index.search(q_embedding, k)
    return kb_df.iloc[indices[0]]

## 3. Load the base model, then attach the LoRA adapter from Phase 4

This is the key step of this notebook: `PeftModel.from_pretrained` loads the frozen base model, then layers your trained adapter weights on top — the same combination trained in Phase 4, now also getting retrieved context at inference time.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(adapter_repo)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)
model = PeftModel.from_pretrained(base_model, adapter_repo)
print("Fine-tuned model (base + LoRA adapter) loaded.")

## 4. Write the fine-tuned + RAG generation function

Same RAG prompt-building approach as Phase 3, but now running through the fine-tuned model instead of the raw base model.

In [ ]:
import time

def build_rag_prompt(question, retrieved):
    context_block = "\n\n".join(
        f"Reference {i+1}:\nQ: {row['instruction']}\nA: {row['response']}"
        for i, (_, row) in enumerate(retrieved.iterrows())
    )
    return (
        "You are a helpful customer support assistant. "
        "Use the reference Q&A pairs below to answer the customer's question. "
        "If the references don't cover the question, say what you can and note the limitation — "
        "do not invent policy details that aren't in the references.\n\n"
        f"{context_block}"
    )

def generate_finetuned_rag_answer(question, k=3, max_new_tokens=150):
    retrieved = retrieve(question, k=k)
    system_prompt = build_rag_prompt(question, retrieved)

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    start = time.time()
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
        )
    latency = time.time() - start

    generated = output[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(generated, skip_special_tokens=True)
    return answer.strip(), latency, retrieved["instruction"].tolist()

# sanity check
test_answer, test_latency, sources = generate_finetuned_rag_answer("How can I get a refund for my order?")
print(f"Answer: {test_answer}\n\nLatency: {test_latency:.2f}s\nSources: {sources}")

## 5. Run over all eval questions

In [ ]:
results = []
for i, row in eval_df.iterrows():
    question = row["instruction"]
    answer, latency, sources = generate_finetuned_rag_answer(question)
    results.append({
        "question": question,
        "intent": row.get("intent", ""),
        "finetuned_rag_answer": answer,
        "finetuned_rag_latency_sec": round(latency, 3),
        "retrieved_sources": " | ".join(sources),
    })
    print(f"[{i+1}/{len(eval_df)}] done")

finetuned_rag_df = pd.DataFrame(results)
finetuned_rag_df.to_csv("finetuned_rag_results.csv", index=False)
finetuned_rag_df.head()

## 6. Also run the fine-tuned model WITHOUT RAG (Variant 3) here for convenience

You built the fine-tuning logic in Phase 4, but running the eval loop for Variant 3 here — right next to Variant 4 — makes it easy to keep the two straight and compare them directly.

In [ ]:
SYSTEM_PROMPT = (
    "You are a helpful customer support assistant. "
    "Answer the customer's question clearly and concisely."
)

def generate_finetuned_answer(question, max_new_tokens=150):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    start = time.time()
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    latency = time.time() - start

    generated = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip(), latency

results = []
for i, row in eval_df.iterrows():
    question = row["instruction"]
    answer, latency = generate_finetuned_answer(question)
    results.append({
        "question": question,
        "intent": row.get("intent", ""),
        "finetuned_answer": answer,
        "finetuned_latency_sec": round(latency, 3),
    })
    print(f"[{i+1}/{len(eval_df)}] done")

finetuned_df = pd.DataFrame(results)
finetuned_df.to_csv("finetuned_results.csv", index=False)
finetuned_df.head()

## 7. Push both results files to Hugging Face

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
for fname in ["finetuned_results.csv", "finetuned_rag_results.csv"]:
    api.upload_file(
        path_or_fileobj=fname,
        path_in_repo=f"results/{fname}",
        repo_id=dataset_repo,
        repo_type="dataset",
    )
print("Uploaded both results files.")

## Done — Phase 5 checklist

- [ ] Base model loaded, LoRA adapter attached via `PeftModel.from_pretrained`
- [ ] RAG pipeline (same FAISS index) layered on top
- [ ] All 4 eval questions run through Variant 4 (fine-tuned + RAG)
- [ ] Variant 3 (fine-tuned, no RAG) also run for direct comparison
- [ ] Both results files pushed to Hugging Face

## What you now have
All four variants' results are sitting in your Hugging Face dataset repo under `results/`:
- `baseline_results.csv` (Variant 1)
- `rag_results.csv` (Variant 2)
- `finetuned_results.csv` (Variant 3)
- `finetuned_rag_results.csv` (Variant 4)

## What to look at before moving on
Pick 2-3 eval questions and read all four answers side by side. This is the moment the whole project has been building toward — you should be able to see a real difference in specificity and grounding as you move from Variant 1 through Variant 4. These 2-3 examples are your best live-demo material for the call.

## What to be able to explain on the call
- **Why layer RAG on top of fine-tuning instead of picking one?** Fine-tuning changes *how* the model responds (tone, structure, domain vocabulary), while RAG changes *what facts* it has access to at answer time. They solve different problems, so combining them addresses both at once.
- **Is Variant 4 always better?** Not necessarily by every metric — it may be slightly slower than Variant 3 alone (retrieval adds a step), and if the required answer wasn't in the training data or the knowledge base, both approaches still limit it. This nuance is exactly what Phase 6's evaluation harness will measure precisely instead of relying on eyeballing.

Next: Phase 6 — the evaluation harness that scores all four variants on relevance, faithfulness, and latency.